In [3]:
###########################################################
# Generate YOLO example dataset
###########################################################

import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from PIL import Image
import random
from skimage.draw import polygon
import math

###########################################################
# Dataset folders
###########################################################

here = os.getcwd().replace("\\", "/")

parent_folder = here + "/Yolo_test_shapes"
train_folder = parent_folder + "/train"
val_folder = parent_folder + "/val"

image_train_folder = train_folder + "/images"
label_train_folder = train_folder + "/labels"

image_val_folder = val_folder + "/images"
label_val_folder = val_folder + "/labels"

###########################################################
# Create folders if required
###########################################################

os.makedirs(image_train_folder, exist_ok=True)
os.makedirs(label_train_folder, exist_ok=True)

os.makedirs(image_val_folder, exist_ok=True)
os.makedirs(label_val_folder, exist_ok=True)

###########################################################
# Create YOLO dataset YAML
###########################################################

yaml_text = f"""path: "{parent_folder}"
train: train/images
val: val/images

nc: 2

names:
  0: circle
  1: square
"""

with open("trainingdata_shapes.yaml", "w") as file:
    file.write(yaml_text)

###########################################################
# Dataset settings
###########################################################

data = []

width = 512
number = 100

image_base = np.zeros((width, width))

i = 1

###########################################################
# Generate images
###########################################################

while i <= number:

    ###########################################################
    # Create blank image
    ###########################################################

    image_base = np.zeros((width, width)) + 230

    image_base_rgb = np.stack((image_base,) * 3, axis=-1)

    image_base_rgb[:, :] = [230, 230, 230]

    mask_base = np.zeros((width, width)) + 1

    ###########################################################
    # Random shape size and position
    ###########################################################

    radius = round((np.random.rand() + 0.1) * width / 5) + 5

    x_pos = round((np.random.rand() - 0.5) * width * 0.75 + width / 2)

    y_pos = round((np.random.rand() - 0.5) * width * 0.75 + width / 2)

    ###########################################################
    # Randomly select circle or square
    ###########################################################

    random_int = random.randint(0, 1)

    ###########################################################
    # Circle
    ###########################################################

    if random_int == 0:

        y, x = np.meshgrid(np.arange(1, width + 1), np.arange(1, width + 1))

        distance = np.sqrt((y - y_pos) ** 2 + (x - x_pos) ** 2)

        image_base_rgb[distance <= radius] = [0, 0, 0]

        mask_base[distance <= radius] = 2

        shape = "circle"

        area = round(3.14 * radius * radius)

        surface2d = round(radius * 3.14)

        volume = round(4 / 3 * 3.14 * radius * radius * radius)

    ###########################################################
    # Square coordinates
    ###########################################################

    x_start = max(0, x_pos - round(radius))
    x_end = min(width, x_pos + round(radius))

    y_start = max(0, y_pos - round(radius))
    y_end = min(width, y_pos + round(radius))

    ###########################################################
    # Square
    ###########################################################

    if random_int == 1:

        if x_start < x_end and y_start < y_end:

            image_base_rgb[x_start:x_end, y_start:y_end, :] = [0, 0, 0]

            mask_base[x_start:x_end, y_start:y_end] = 2

            shape = "square"

            area = round(radius * radius)

            surface2d = round(radius * 4)

            volume = round(radius * radius * radius)

    ###########################################################
    # Convert image
    ###########################################################

    image = Image.fromarray(image_base_rgb.astype(np.uint8))

    ###########################################################
    # Create YOLO label
    ###########################################################

    label_string = str(random_int) + " " + str(y_pos / width) + " " + str(x_pos / width) + " " + str(radius * 2 / width) + " " + str(radius * 2 / width)

    ###########################################################
    # Random training or validation split
    ###########################################################

    trainvsval = random.randint(1, 2)

    ###########################################################
    # Training sample
    ###########################################################

    if trainvsval == 1:

        name_label = label_train_folder + f"/image_{i}.txt"

        with open(name_label, "w") as file:
            file.write(label_string)

        image.save(image_train_folder + f"/image_{i}.png")

    ###########################################################
    # Validation sample
    ###########################################################

    if trainvsval == 2:

        name_label = label_val_folder + f"/image_{i}.txt"

        with open(name_label, "w") as file:
            file.write(label_string)

        image.save(image_val_folder + f"/image_{i}.png")

    ###########################################################
    # Next sample
    ###########################################################

    i += 1

###########################################################
# Finished
###########################################################

print("YOLO example dataset generated")

print("Dataset folder:")
print(parent_folder)

print()

print("Dataset YAML:")
print(here + "/trainingdata_shapes.yaml")

YOLO example dataset generated
Dataset folder:
C:/Users/bowan/Desktop/Git_hub_folder_test/Thesis_code/Tutorial_code/Pure_image_methods/Detection/Yolo_test_shapes

Dataset YAML:
C:/Users/bowan/Desktop/Git_hub_folder_test/Thesis_code/Tutorial_code/Pure_image_methods/Detection/trainingdata_shapes.yaml
